In [ ]:
%matplotlib inline

In [ ]:
import autograd.numpy as np
from autograd import grad, jacobian

In [ ]:
import matplotlib.pyplot as plt
from mpltools import annotation

# Session 1: Differentiable Programming with HIPS/Autograd

<div class="alert alert-block alert-warning" style="background-color: rgb(236,176,146); border: 2px solid rgb(213,104,79); color: rgb(64,64,64);">
    
<b>Set up codespace now. (It will take a while!)</b>

We will show you where to find this notebook in the repo while you wait.

</div>

## Motivation

Differentiable programming is an enabling technology. Given scientific code for computing some quantity, it allows us to generate derivatives of quantities involved in the computation without any need for deriving or hand-coding the derivative expressions involved.

Some motivating examples:
* Computing the **Jacobian** for a nonlinear system.
* Computing the **gradient** required for an ODE- or PDE-constrained optimisation method.
* The **backpropagation** operation used for training machine learning models.
* Computing **Hessians** for uncertainty quantification methods.
* Solving the **adjoint** problems involved in data assimilation methods commonly used for weather forecasting.

## Learning objectives

In today's session we will:

* Get a brief history of automatic differentiation.
* Learn about *forward mode* and *reverse mode*.
* Learn about the *operator overloading* approach.
* Try out the *Autograd* AD tool applied to some test problems.
* Verify the derivatives produced by Autograd both manually and using the *Taylor test*.

## Preparations

#### Terminology

This course introduces the concept of *differentiable programming*, a.k.a. *automatic differentiation (AD)*, or *algorithmic differentiation*. We will use the acronym AD henceforth.

#### Notation

For a differentiable *mathematical* function $f:A\rightarrow\mathbb{R}$ with scalar input (i.e., a single value) from $A\subseteq\mathbb{R}$, we make use of both the Lagrange notation $f'(x)$ and Leibniz notation $\frac{\mathrm{d}f}{\mathrm{d}x}$ for its derivative.

<div class="alert alert-block alert-warning" style="background-color: rgb(236,176,146); border: 2px solid rgb(213,104,79); color: rgb(64,64,64);">
<b>Caution</b> with the physics notation for derivatives $\dot{x}$. It won't always mean what you expect! (See later.)
</div>

Similarly, for $m\in\mathbb{N}$ dimensional, differentiable, vector-valued function $\mathbf{f}:A\rightarrow\mathbb{R}^m$ with scalar input, we have derivative notations $\mathbf{f}'(x)$ and $\frac{\mathrm{d}\mathbf{f}}{\mathrm{d}x}$.

For a differentiable function with vector input (i.e., multiple inputs), we use partial derivative notation. For example, if $f:\mathbb{R}^2\rightarrow\mathbb{R}$ is written as $f=f(x,y)$ then we have the partial derivatives $\frac{\partial f}{\partial x}$ and $\frac{\partial f}{\partial y}$ with respect to first and second components, respectively. We use
$$\nabla f=\left(\frac{\partial f}{\partial x_1},\dots,\frac{\partial f}{\partial x_m}\right)$$
to denote the vector of all such partial derivatives. Similarly for vector-valued functions with multiple inputs.

When it comes to derivatives in code, we use the `_d` notation (for "derivative" or "dot"), which is standard in the AD literature. Its meaning will be described in due course.

## History

* Origins of AD in 1950s.
* However, it found a wider audience in the 1980s, when it became more relevant thanks to advances in both computer power and modern programming languages.
* Forward mode (the subject of this session) was discovered by Wengert in 1964.
* Further developed by Griewank in the late 1980s.

<div style="text-align: center;">
  <img src="images/Wengert.png" width="600" style="display: block; margin-left: auto; margin-right: auto;"/>
  <strong>Figure 1:</strong> Header of <a href="https://doi.org/10.1145/355586.364791">(R. E. Wengert, 1964)</a>.
</div>

## Idea

The idea of AD is to **treat a model as a sequence of elementary instructions** (e.g., addition, multiplication, exponentiation). Here a *model* could be a function or subroutine, code block, or a whole program. Elementary operations are well-understood and their derivatives are known. As such, the derivative of the whole model may be computed by composing the derivatives of each operation using the *chain rule*.

#### Recap on A-level maths: the Chain Rule

Consider two composable, differentiable (mathematical) functions, $f$ and $g$, with composition $h=f\circ g$. By definition, this means
$$h(x)=(f\circ g)(x)=g(f(x)).$$

Then the *chain rule* states that the derivative of $h$ may be computed in terms of the derivatives of $f$ and $g$ using the formula
$$h'(x)=(f\circ g)'(x)=(f\circ g')(x)\,f'(x)=g'(f(x))\,f'(x).$$

Equivalently, in Leibniz notation:
$$\frac{\mathrm{d}h}{\mathrm{d}x}=\frac{\mathrm{d}g}{\mathrm{d}f}\frac{\mathrm{d}f}{\mathrm{d}x}.$$

For variables with multiple arguments, the result is equivalent for each partial derivative, e.g.,
$$\frac{\partial h}{\partial x}=\frac{\partial g}{\partial f}\frac{\partial f}{\partial x}.$$

## Example: ODE-constrained optimisation

Consider the scalar ordinary differential equation (ODE)
$$
    \frac{\mathrm{d}u}{\mathrm{d}t}=f(u),\quad u(0)=u_0,
$$
where $t\in[0,T]$ is the time variable, $T>0$ is the end time, and $u_0\in\mathbb{R}$ is the initial condition. Given some $f:A\rightarrow\mathbb{R}$ with $A\subseteq\mathbb{R}$, we seek to solve the ODE for $u:[0,T]\rightarrow\mathbb{R}$.

For simplicity, let's consider the ODE
$$
    \frac{\mathrm{d}u}{\mathrm{d}t}=u,\quad u(0)=1,
$$
i.e., $f(u)=u$.

<div class="alert alert-block alert-info" style="background-color: rgb(195,223,220); border: 2px solid rgb(157,204,199); color: rgb(0,100,100);">
<b>Optional exercise</b>
    
Convince yourself that the analytical solution of the ODE is $u(t)=\mathrm{e}^t$.

<b>Solutions</b>
    
<details>

Plugging $u(t)=\mathrm{e}^t$ into the LHS gives $\frac{\mathrm{d}u}{\mathrm{d}t}=\mathrm{e}^t=u$, which satisfies the ODE. Checking the initial condition, we have $u(0)=\mathrm{e}^0=1$, which also satisfies.

</details>
</div>

The initial condition can be implemented as

In [ ]:
def initial_condition():
    """
    Apply the initial condition for the ODE initial value problem
        du/dt = u, u(0) = 1
    """
    u0 = 1.0
    return u0

## ODE example: forward vs backward Euler

You're probably aware of the simplest example of an explicit timestepping method to approximate the solution of the ODE. This is the *forward Euler* (a.k.a. explicit Euler):
$$
    \frac{u_{k}-u_{k-1}}{\Delta t}=f(u_{k-1}),
$$
for $k\in\mathbb{N}$ and some timestep $\Delta t>0$.

You're probably also aware that the simplest example of an implicit timestepping method to approximate the solution of the ODE is *backward Euler* (a.k.a. implicit Euler):
$$
    \frac{u_{k}-u_{k-1}}{\Delta t}=f(u_k),
$$
for $k\in\mathbb{N}$ and some timestep $\Delta t>0$.

These are actually special cases of a more general *theta-method*,
$$
    \frac{u_{k}-u_{k-1}}{\Delta t}=(1-\theta)f(u_{k-1})+\theta f(u_k),
$$
where $\theta\in[0,1]$.

## ODE example: applying forward and backward Euler

Using the method above, our problem reads
$$
    \frac{u_{k}-u_{k-1}}{\Delta t}=(1-\theta)u_{k-1}+\theta u_k,
$$
which can be rearranged to give
$$
    u_{k}=\frac{1+\Delta t(1-\theta)}{1-\Delta t\theta}u_{k-1}.
$$
We can implement this using a Python function as

In [ ]:
def theta_step(u_, dt, theta):
    """
    Take a single iteration of a theta method for solving the ODE initial value problem
        du/dt = u, u(0) = 1
    """
    u = u_ * (1.0 + dt * (1.0 - theta)) / (1.0 - dt * theta)
    return u

We can define the $\theta$-method in terms of the `initial_condition` and `theta_step` functions as

In [ ]:
def theta_method(theta):
    """
    Solve the ODE initial value problem
        du/dt = u, u(0) = 1
    using a theta timestepping method, returning the solution trajectory.
    """
    t = 0.0
    dt = 0.1
    end_time = 1.0
    u0 = initial_condition()

    # Timestepping loop
    trajectory = [u0]
    u_ = u0
    while t < end_time - 1.0e-05:
        u = theta_step(u_, dt, theta)
        u_ = u
        t += dt
        trajectory.append(u)
    return trajectory

Forward Euler corresponds to $\theta=0$ and Backward Euler corresponds to $\theta=1$ so we can apply them to the problem with

In [ ]:
forward = theta_method(0.0)
backward = theta_method(1.0)

times = np.linspace(0, 1, len(forward))

fig, axes = plt.subplots()
axes.plot(times, np.exp(times), "-", color="k", label="Analytical solution")
axes.plot(times, forward, "--x", label="Forward Euler")
axes.plot(times, backward, ":o", label="Backward Euler")
axes.legend()
axes.grid()

## ODE example: source transformation

As we see from the plot above, the forward Euler method tends to underestimate the solution, whereas the backward Euler method tends to overestimate it. Let's try to optimise the value of $\theta$ to best match the solution using a gradient-based optimisation method. To do that, we first need the gradient. **AD enables us to do this automatically.**

The optimisation problem we seek to solve is to minimise some error measure $J$ for the approximation of $u$ by varying $\theta$. That is,
$$
    \min_{\theta\in[0,1]}J(u;\theta).
$$
where the notation $J(u;\theta)$ refers to the implicit dependence of the solution approximation $u$ on $\theta$.

<div class="alert alert-block alert-warning" style="background-color: rgb(236,176,146); border: 2px solid rgb(213,104,79); color: rgb(64,64,64);">
    
<b>Note</b> Forward and backward Euler are first-order accurate methods. By optimising the $\theta$ parameter, we can arrive at a second-order accurate method.

</div>

## ODE example: optimisation with gradient descent

Let's solve this ODE problem with one of the simplest gradient-based optimisation approaches: gradient descent. This amounts to an initial guess $\theta_0$, followed by iterative updates
$$
    \theta_{k+1}=\theta_k+\alpha\:p_k,
$$
where $\alpha>0$ is the step length and $p_k$ is the descent direction. For gradient descent, we simply take
$$
    p_k=-\frac{\mathrm{d}J_k}{\mathrm{d}\theta_k}.
$$

Since we know the analytical solution for this problem, we may make an 'artificial' choice of cost function such as
$$
    J(u;\theta)=(u(1)-\mathrm{e}^1)^2,
$$
where here $\mathrm{e}^1$ is the analytical solution at the end time $t=1$. We can implement this as the Python function

In [ ]:
def cost_function(theta):
    """
    Cost function evaluating the l2 error at the end time against the analytical solution u(t)=exp(t)
    """
    u = theta_method(theta)[-1]
    e = np.exp(1.0)
    return (u - e) ** 2

We want to track the convergence progress of the gradient descent method so let's create arrays to store the control and cost function values.

In [ ]:
controls = []
costs = []

A simple implementation of the gradient descent method can be found in the following code block, although there is a missing piece.

<div class="alert alert-block alert-info" style="background-color: rgb(195,223,220); border: 2px solid rgb(157,204,199); color: rgb(0,100,100);">
    
<b>Exercise</b>
    
You will also need to differentiate the cost function with respect to $\theta$. Replace the `# TODO` comment with your implementation.

<b>Solution</b>
    
<details>

```python
Jd = grad(cost_function)(theta)
```

</details>

</div>

<div class="alert alert-block alert-warning" style="background-color: rgb(236,176,146); border: 2px solid rgb(213,104,79); color: rgb(64,64,64);">
    
<b>Hint</b> There are two derivatives being considered here. The ODE is a model, which involves the derivative of the model solution on the left hand side. This derivative is being approximated using a numerical method, not computed using AD. The ODE model is parametrised by $\theta$. The gradient descent method considers the gradient of the cost function (defined in terms of model outputs) with respect to the $\theta$ parameter. We want to use AD to understand the relationship between the model's accuracy (in some sense) and its parameters, not to evaluate derivatives that appear in the model.

</div>

In [ ]:
def gradient_descent(maxiter=1000, gtol=1.0e-05, dtol=1.1, alpha=0.10):
    """
    Function for optimising the theta parameter for a theta timestepping method for solving the ODE
        du/dt = u, u(0)=1
    using gradient descent.
    """
    # Start from forward Euler
    theta = 0.0

    for i in range(maxiter):
        J = cost_function(theta)
        # Jd = # TODO
        Jd = grad(cost_function)(theta)
        
        controls.append(theta)
        costs.append(J)
        
        # Convergence and divergence checks
        if i == 0:
            J_init = J
        elif abs(Jd / Jd_) < gtol:
            print(f"Converged in {i+1} iterations due to gradient convergence")
            return theta
        elif abs(J / J_init) > dtol:
            raise RuntimeError(f"Detected divergence after {i+1} iterations")
        Jd_ = Jd

        # Take a step in the descent direction
        p = -Jd
        theta += alpha * p
    # raise RuntimeError("Reached maximum iteratons without convergence")  # FIXME: make more robust
    return theta

We can then run the gradient descent algorithm and plot it's progress as follows.

In [ ]:
theta_opt = gradient_descent()

costs[0] = np.nan  # Remove the first entry because it's uninitialised garbage

fig, axes = plt.subplots(ncols=2, figsize=(12, 5))
axes[0].loglog(costs, "--", label="Cost function value")
axes[0].legend()
axes[0].grid()
axes[1].plot(controls, "--", label="Control value")
axes[1].legend()
axes[1].grid()

This looks promising! Let's examine the solution trajectory for the optimised $\theta$ parameter to check it does a better job than Forward Euler and Backward Euler.

In [ ]:
optimised = theta_method(theta_opt)

fig, axes = plt.subplots()
axes.plot(times, np.exp(times), "-", color="k", label="Analytical solution")
axes.plot(times, forward, "--x", label="Forward Euler")
axes.plot(times, backward, ":o", label="Backward Euler")
axes.plot(times, optimised, "-.^", label=rf"Optimised ($\theta={theta_opt:.4f}$)")
axes.legend()
axes.grid()

As we might hope, the optimised value of $\theta$ gives a much better approximation.